In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}

Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai__Key = '' # @param {type:"string", placeholder:"Get Civitai key: https://civitai.com/user/account"}
HF_Read_Token = '' # @param {type:"string", placeholder:"Get HF token: https://huggingface.co/settings/tokens (optional)"}
Mount_GDrive = 'No' # @param ["No", "Yes"]

from pathlib import Path
import json
import os
import shlex
import subprocess

SEGS_REPO = 'https://github.com/N3iKos/segsmaker-fast'
SEG_DRIVE_ROOT = Path('/content/drive/MyDrive/Segsmaker')
SEG_DRIVE_ENABLED = Mount_GDrive == 'Yes'

if SEG_DRIVE_ENABLED:
    from google.colab import drive
    drive.mount('/content/drive')
    SEG_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

!curl -sLo /content/setup.py https://github.com/N3iKos/segsmaker-fast/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai__Key" --hf_read_token="$HF_Read_Token"

def seg_drive_dir(name):
    folder = SEG_DRIVE_ROOT / name
    if SEG_DRIVE_ENABLED:
        folder.mkdir(parents=True, exist_ok=True)
    return folder

def seg_link_dir(name, runtime_path):
    if not SEG_DRIVE_ENABLED or runtime_path is None:
        return
    runtime_path = Path(runtime_path)
    runtime_path.mkdir(parents=True, exist_ok=True)
    link = runtime_path / f'drive-{name}'
    target = seg_drive_dir(name)
    if link.exists() or link.is_symlink():
        return
    try:
        link.symlink_to(target, target_is_directory=True)
        print(f'[Drive] linked {link} -> {target}')
    except Exception as exc:
        print(f'[Drive] link skipped for {name}: {exc}')

def seg_download_many(items, target_dir, drive_name, parallel=False, max_workers=3, load_from_drive=False):
    target_dir = Path(target_dir) if target_dir else None
    if target_dir is None:
        print(f'[skip] target path for {drive_name} is not available for this WebUI')
        return []
    target_dir.mkdir(parents=True, exist_ok=True)
    drive_dir = seg_drive_dir(drive_name) if (SEG_DRIVE_ENABLED and load_from_drive) else None
    return download_many(
        items,
        target_dir=str(target_dir),
        parallel=parallel,
        max_workers=max_workers,
        load_from_drive=bool(load_from_drive and SEG_DRIVE_ENABLED),
        drive_dir=str(drive_dir) if drive_dir else None,
    )

def seg_clone_repos(urls, target_dir, parallel=False, max_workers=3):
    urls = [url.strip() for url in urls if url.strip()]
    if not urls:
        print('  no extension/custom-node URLs')
        return []
    target_dir = Path(target_dir) if target_dir else None
    if target_dir is None:
        print('[skip] extensions path is not available for this WebUI')
        return []
    target_dir.mkdir(parents=True, exist_ok=True)

    def clone_one(url):
        command = shlex.split(url if url.startswith('git clone ') else f'git clone {url}')
        subprocess.run(command, cwd=str(target_dir), check=True)
        return command[-1]

    if not parallel or len(urls) == 1:
        results = []
        for index, url in enumerate(urls, 1):
            result = clone_one(url)
            print(f'  [{index:>2}/{len(urls)}] ? {result}')
            results.append(result)
        return results

    from concurrent.futures import ThreadPoolExecutor, as_completed
    results = []
    with ThreadPoolExecutor(max_workers=max(1, min(int(max_workers or 1), len(urls)))) as executor:
        futures = {executor.submit(clone_one, url): url for url in urls}
        for index, future in enumerate(as_completed(futures), 1):
            try:
                result = future.result()
                print(f'  [{index:>2}/{len(urls)}] ? {result}')
                results.append(result)
            except Exception as exc:
                print(f'  [{index:>2}/{len(urls)}] ? {futures[future]}: {exc}')
    return results

if SEG_DRIVE_ENABLED:
    for name, runtime_path in {
        'checkpoint': globals().get('CKPT'),
        'lora': globals().get('LORA'),
        'vae': globals().get('VAE'),
        'embeddings': globals().get('Embeddings'),
        'upscalers': globals().get('Upscalers'),
        'unet': globals().get('UNET'),
        'clip': globals().get('CLIP'),
        'text_encoder': globals().get('TE'),
    }.items():
        seg_link_dir(name, runtime_path)

    output_dir = SEG_DRIVE_ROOT / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    output_dir.mkdir(parents=True, exist_ok=True)
    try:
        if WebUI_Output.exists() and not WebUI_Output.is_symlink():
            print(f'[Drive] output path already exists, leaving it unchanged: {WebUI_Output}')
        elif not WebUI_Output.exists():
            WebUI_Output.symlink_to(output_dir, target_is_directory=True)
    except Exception as exc:
        print(f'[Drive] output link skipped: {exc}')


In [ ]:
# @title <b><font color='orange'>Model Downloader</font></b> {"display-mode":"form"}

Checkpoint_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_4 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Lora_5 = '' # @param {type:"string", placeholder:"URL or leave empty"}
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty"}
Load_from_Drive = True # @param {type:"boolean"}
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:10, step:1}

checkpoint_items = [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]
lora_items = [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]
vae_items = [VAE_URL]

seg_download_many(checkpoint_items, globals().get('CKPT'), 'checkpoint', Parallel_Download, Max_Workers, Load_from_Drive)
seg_download_many(lora_items, globals().get('LORA'), 'lora', Parallel_Download, Max_Workers, Load_from_Drive)
seg_download_many(vae_items, globals().get('VAE'), 'vae', Parallel_Download, Max_Workers, Load_from_Drive)


In [ ]:
# @title <b><font color='orange'>Extra Assets</font></b> {"display-mode":"form"}

Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Embedding_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Load_from_Drive = True # @param {type:"boolean"}
Assets_Parallel_Download = True # @param {type:"boolean"}
Assets_Max_Workers = 3 # @param {type:"slider", min:1, max:10, step:1}

extensions = [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5]
embeddings = [Embedding_1, Embedding_2, Embedding_3]
upscalers = [Upscaler_1, Upscaler_2, Upscaler_3]

seg_clone_repos(extensions, globals().get('Extensions'), Assets_Parallel_Download, Assets_Max_Workers)
seg_download_many(embeddings, globals().get('Embeddings'), 'embeddings', Assets_Parallel_Download, Assets_Max_Workers, Load_from_Drive)
seg_download_many(upscalers, globals().get('Upscalers'), 'upscalers', Assets_Parallel_Download, Assets_Max_Workers, Load_from_Drive)


In [ ]:
# @title <b><font color='orange'>FLUX Model Downloader</font></b> {"display-mode":"form"}

FLUX_Variant = 'FLUX.1-schnell' # @param ["FLUX.1-schnell", "FLUX.1-dev"]
FLUX_Unet = '' # @param {type:"string", placeholder:"Custom Unet URL or leave empty"}
FLUX_Clip_L = '' # @param {type:"string", placeholder:"Custom Clip L URL or leave empty"}
FLUX_T5XXL = '' # @param {type:"string", placeholder:"Custom T5XXL URL or leave empty"}
FLUX_VAE = '' # @param {type:"string", placeholder:"Custom VAE URL or leave empty"}
Load_from_Drive = True # @param {type:"boolean"}
Parallel_FLUX_Download = True # @param {type:"boolean"}
FLUX_Max_Workers = 2 # @param {type:"slider", min:1, max:6, step:1}

FLUX_DEFAULTS = {
    'FLUX.1-schnell': {
        'unet': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/flux1-schnell.safetensors',
        'clip_l': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        't5xxl': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors',
        'vae': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors flux_ae.safetensors',
    },
    'FLUX.1-dev': {
        'unet': 'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/flux1-dev.safetensors',
        'clip_l': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
        't5xxl': 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors',
        'vae': 'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/ae.safetensors flux_ae.safetensors',
    },
}

def choose(custom, key):
    return custom.strip() or FLUX_DEFAULTS[FLUX_Variant][key]

flux_unet_dir = globals().get('UNET') or globals().get('CKPT')
flux_clip_dir = globals().get('CLIP')
flux_t5_dir = globals().get('TE') or globals().get('CLIP')
flux_vae_dir = globals().get('VAE')

seg_download_many([choose(FLUX_Unet, 'unet')], flux_unet_dir, 'flux-unet', Parallel_FLUX_Download, FLUX_Max_Workers, Load_from_Drive)
seg_download_many([choose(FLUX_Clip_L, 'clip_l')], flux_clip_dir, 'flux-clip', Parallel_FLUX_Download, FLUX_Max_Workers, Load_from_Drive)
seg_download_many([choose(FLUX_T5XXL, 't5xxl')], flux_t5_dir, 'flux-text-encoder', Parallel_FLUX_Download, FLUX_Max_Workers, Load_from_Drive)
seg_download_many([choose(FLUX_VAE, 'vae')], flux_vae_dir, 'flux-vae', Parallel_FLUX_Download, FLUX_Max_Workers, Load_from_Drive)


In [ ]:
''' Controlnet '''
%run $Controlnet_Widget


In [ ]:
# @title <b><font color='orange'>Launcher WebUI</font></b> {"display-mode":"form"}

Software = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Ngrok_Token = '' # @param {type:"string", placeholder:"Optional NGROK token"}
Zrok_Token = '' # @param {type:"string", placeholder:"Optional ZROK token"}
Extra_Args = '' # @param {type:"string", placeholder:"Additional launch arguments"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
Skip_Widget = False # @param {type:"boolean"}

print('Select the same WebUI that you installed in the first cell.')
installed = None
try:
    mark_path = Path(HOMEPATH) / 'gutris1/marking.json'
    if mark_path.exists():
        installed = json.loads(mark_path.read_text()).get('ui')
except Exception:
    installed = globals().get('Webui')

if installed and Software != installed:
    print(f'[warning] Installed WebUI is {installed}, but launcher selection is {Software}. The installed WebUI will be used by segsmaker.py.')

if Skip_Widget:
    print('[warning] Skip_Widget is accepted for form compatibility, but Colab/Kaggle launcher has no interactive widget to skip.')

args = []
if Skip_ComfyUI_Check:
    args.append('--skip-comfyui-check')
if Ngrok_Token.strip():
    args.append(f'--N={Ngrok_Token.strip()}')
if Zrok_Token.strip():
    args.append(f'--Z={Zrok_Token.strip()}')
if Extra_Args.strip():
    args.extend(shlex.split(Extra_Args.strip()))

%cd -q $WebUI
run_line = 'segsmaker.py ' + ' '.join(shlex.quote(arg) for arg in args)
get_ipython().run_line_magic('run', run_line)
